In [0]:
from pyspark.sql import functions as F

PATH = "/Volumes/aviation/bronze/archives/vra/*.csv"
SCHEMA = "aviation.bronze.vra"

In [0]:
df = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skiprows", 1)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "utf-8")
    .option("mode", "PERMISSIVE")
    .load(PATH)
)

print("Colunas lidas do arquivo:")
for c in df.columns:
    print(f"  {c!r}")

In [0]:
from unicodedata import normalize
import re

df = df.toDF(
    *[
        re.sub(r"[^\w]", "", normalize("NFD", c))
        .lower()
        .replace(" ", "_")
        .encode("ascii", "ignore")
        .decode("utf-8") for c in df.columns
    ]
)

In [0]:
df = df.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

In [0]:
(
    df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SCHEMA)
)


In [0]:
display(
    spark.sql(
        f"""
            SELECT _arquivo_origem, COUNT('*') AS linhas, MAX
                (_ingerido_em) AS ingerido_em
            FROM {SCHEMA}
            GROUP BY _arquivo_origem
            ORDER BY _arquivo_origem
        """
    )
)